In [1]:
!pip install -q xgboost imbalanced-learn pytorch-tabnet shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [4]:
from google.colab import files

uploaded = files.upload()

Saving water_potability.csv to water_potability.csv


In [5]:
import os

print(os.listdir('/content'))

['.config', 'water_potability.csv', 'sample_data']


In [6]:
import pandas as pd

df = pd.read_csv('/content/water_potability.csv')

print("Dataset shape:", df.shape)

Dataset shape: (3276, 10)


In [7]:
df.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ph               2785 non-null   float64
 1   Hardness         3276 non-null   float64
 2   Solids           3276 non-null   float64
 3   Chloramines      3276 non-null   float64
 4   Sulfate          2495 non-null   float64
 5   Conductivity     3276 non-null   float64
 6   Organic_carbon   3276 non-null   float64
 7   Trihalomethanes  3114 non-null   float64
 8   Turbidity        3276 non-null   float64
 9   Potability       3276 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 256.1 KB


In [9]:
print(df.isnull().sum())

ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64


In [10]:
print(df['Potability'].value_counts())

Potability
0    1998
1    1278
Name: count, dtype: int64


In [11]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [12]:
print(df.describe().T)

                  count          mean          std         min           25%  \
ph               2785.0      7.080795     1.594320    0.000000      6.093092   
Hardness         3276.0    196.369496    32.879761   47.432000    176.850538   
Solids           3276.0  22014.092526  8768.570828  320.942611  15666.690297   
Chloramines      3276.0      7.122277     1.583085    0.352000      6.127421   
Sulfate          2495.0    333.775777    41.416840  129.000000    307.699498   
Conductivity     3276.0    426.205111    80.824064  181.483754    365.734414   
Organic_carbon   3276.0     14.284970     3.308162    2.200000     12.065801   
Trihalomethanes  3114.0     66.396293    16.175008    0.738000     55.844536   
Turbidity        3276.0      3.966786     0.780382    1.450000      3.439711   
Potability       3276.0      0.390110     0.487849    0.000000      0.000000   

                          50%           75%           max  
ph                   7.036752      8.062066     14.000000  

In [13]:
correlation = df.corr(numeric_only=True)['Potability'].sort_values(ascending=False)

print(correlation)

Potability         1.000000
Solids             0.033743
Chloramines        0.023779
Trihalomethanes    0.007130
Turbidity          0.001581
ph                -0.003556
Conductivity      -0.008128
Hardness          -0.013837
Sulfate           -0.023577
Organic_carbon    -0.030001
Name: Potability, dtype: float64


In [14]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (3276, 10)


In [15]:
feature_columns = [col for col in df.columns if col != 'Potability']

for col in feature_columns:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after imputation:")
print(df.isnull().sum())

Missing values after imputation:
ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64


In [16]:
X = df.drop('Potability', axis=1)
y = df['Potability']

print("Features:", X.columns.tolist())
print("Target:", y.name)

Features: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']
Target: Potability


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 2620
Testing samples: 656


In [18]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [19]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

print("10-Fold Stratified Cross Validation created.")

10-Fold Stratified Cross Validation created.


In [20]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

In [21]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Random Forest - 10 Fold CV")

print("Accuracy :", cv_results['test_accuracy'].mean())
print("Precision:", cv_results['test_precision'].mean())
print("Recall   :", cv_results['test_recall'].mean())
print("F1 Score :", cv_results['test_f1'].mean())

Random Forest - 10 Fold CV
Accuracy : 0.6450381679389313
Precision: 0.5480412163871717
Recall   : 0.5215495907100705
F1 Score : 0.5333620845498434


In [22]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss'
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "SVM": SVC(
        kernel='rbf',
        probability=True,
        random_state=42
    )
}

results = []

for name, model in models.items():

    if name == "SVM":
        pipeline = Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('scaler', StandardScaler()),
            ('model', model)
        ])
    else:
        pipeline = Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('model', model)
        ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            'accuracy': 'accuracy',
            'precision': 'precision',
            'recall': 'recall',
            'f1': 'f1'
        },
        n_jobs=-1
    )

    results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1 Score': scores['test_f1'].mean()
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by='F1 Score',
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
3,SVM,0.640076,0.534317,0.638997,0.580479
0,Random Forest,0.645038,0.548041,0.521550,0.533362
1,XGBoost,0.613359,0.503945,0.551923,0.526273
2,Gradient Boosting,0.599618,0.488204,0.552922,0.517860


In [23]:
raw_df = pd.read_csv('/content/water_potability.csv')

X = raw_df.drop('Potability', axis=1)
y = raw_df['Potability']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (2620, 9)
Testing : (656, 9)


In [24]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00


In [25]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss'
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "SVM": SVC(
        kernel='rbf',
        probability=True,
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        iterations=200,
        depth=5,
        learning_rate=0.05,
        verbose=False,
        random_seed=42
    )
}

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():

    steps = [
        ('imputer', SimpleImputer(strategy='median')),
        ('smote', SMOTE(random_state=42))
    ]

    if name == "SVM":
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    pipeline = Pipeline(steps)

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            'accuracy': 'accuracy',
            'precision': 'precision',
            'recall': 'recall',
            'f1': 'f1'
        },
        n_jobs=-1
    )

    results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1 Score': scores['test_f1'].mean()
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by='F1 Score',
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1 Score
0,SVM,0.638931,0.533255,0.637036,0.579035
1,CatBoost,0.637023,0.532600,0.582267,0.555537
2,Random Forest,0.648092,0.553384,0.516686,0.533403
3,XGBoost,0.610305,0.500733,0.550999,0.523951
4,Gradient Boosting,0.598092,0.486762,0.551923,0.516397


In [26]:
overfitting_results = []

for name, model in models.items():

    steps = [
        ('imputer', SimpleImputer(strategy='median')),
        ('smote', SMOTE(random_state=42))
    ]

    if name == "SVM":
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    pipeline = Pipeline(steps)

    # Train on complete training set
    pipeline.fit(X_train, y_train)

    # Training predictions
    train_pred = pipeline.predict(X_train)

    train_f1 = f1_score(y_train, train_pred)

    # CV F1 from previous results
    cv_f1 = results_df[
        results_df['Model'] == name
    ]['F1 Score'].values[0]

    overfitting_results.append({
        'Model': name,
        'Training F1': train_f1,
        'CV F1': cv_f1,
        'Difference': train_f1 - cv_f1
    })

overfit_df = pd.DataFrame(overfitting_results)

overfit_df.sort_values(
    by='CV F1',
    ascending=False
).reset_index(drop=True)

,Model,Training F1,CV F1,Difference
0,SVM,0.696600,0.579035,0.117565
1,CatBoost,0.736045,0.555537,0.180508
2,Random Forest,1.000000,0.533403,0.466597
3,XGBoost,0.752708,0.523951,0.228757
4,Gradient Boosting,0.669169,0.516397,0.152773


In [27]:
rf_corrected = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ))
])

In [28]:
rf_corrected.fit(X_train, y_train)

rf_train_pred = rf_corrected.predict(X_train)
rf_cv_scores = cross_validate(
    rf_corrected,
    X_train,
    y_train,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

print("Corrected Random Forest")
print("-----------------------")
print("Training F1:", f1_score(y_train, rf_train_pred))
print("CV F1:", rf_cv_scores['test_score'].mean())
print(
    "Difference:",
    f1_score(y_train, rf_train_pred) -
    rf_cv_scores['test_score'].mean()
)

Corrected Random Forest
-----------------------
Training F1: 0.793070259865255
CV F1: 0.5332541033663704
Difference: 0.2598161564988847


In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Build the final SVM pipeline
final_svm = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('model', SVC(
        kernel='rbf',
        probability=True,
        random_state=42
    ))
])

# Train on the complete training set
final_svm.fit(X_train, y_train)

# Predict on untouched test set
y_test_pred = final_svm.predict(X_test)

# Evaluation
print("Final Test Evaluation - SVM")
print("---------------------------")
print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall   :", recall_score(y_test, y_test_pred))
print("F1 Score :", f1_score(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Final Test Evaluation - SVM
---------------------------
Accuracy : 0.6021341463414634
Precision: 0.49158249158249157
Recall   : 0.5703125
F1 Score : 0.5280289330922242

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.62      0.66       400
           1       0.49      0.57      0.53       256

    accuracy                           0.60       656
   macro avg       0.59      0.60      0.59       656
weighted avg       0.61      0.60      0.61       656

Confusion Matrix:
[[249 151]
 [110 146]]
